# Pork rasher defect detection: Colab workflow

Thin notebook: all logic lives in the `porkrasher` package. Use a GPU runtime (Runtime > Change runtime type > T4 GPU).


In [ ]:
# EDIT THESE TWO
REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO.git"
ZIP_PATH = "/content/drive/MyDrive/YOUR_FOLDER/dataset.zip"


In [ ]:
!git clone $REPO_URL repo
%cd repo
!pip install -q -e .


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from porkrasher.data import extract_dataset, write_data_yaml
root = extract_dataset(ZIP_PATH, '/content/dataset', overwrite=True)
data_yaml = write_data_yaml(root)
print(root, data_yaml)


In [ ]:
!python -m porkrasher.data --root /content/dataset


In [ ]:
from porkrasher.visualise import plot_samples
fig = plot_samples('/content/dataset/train/images', '/content/dataset/train/labels', n=4, seed=0)


## Training (selected model: YOLOv8n; comparison: YOLOv8s)

In [ ]:
from porkrasher.train import train
best_n = train('yolov8n.pt', data_yaml, epochs=100, name='yolov8n_100ep')
print(best_n)


In [ ]:
best_s = train('yolov8s.pt', data_yaml, epochs=100, name='yolov8s_100ep')
print(best_s)


## Validation split (use for model selection)

In [ ]:
from porkrasher.evaluate import evaluate, format_results_table, format_per_class_table
res_n = evaluate(best_n, data_yaml, split='val', out_json='results/yolov8n_val.json')
res_s = evaluate(best_s, data_yaml, split='val', out_json='results/yolov8s_val.json')
results = {'yolov8n': res_n, 'yolov8s': res_s}
print(format_results_table(results))
print(format_per_class_table(results, 'map50'))


## Test split

**Warning:** run this once, on the final chosen model only. Do not tune anything after seeing these numbers.

In [ ]:
res_test = evaluate(best_n, data_yaml, split='test', out_json='results/yolov8n_test.json')
print(format_results_table({'yolov8n': res_test}))


In [ ]:
from porkrasher.evaluate import benchmark_inference
from porkrasher.data import list_images
print(benchmark_inference(best_n, list_images('/content/dataset/test/images')))


Copy `best.pt` for both models to Drive so they survive the session.

In [ ]:
!cp {best_n} /content/drive/MyDrive/yolov8n_best.pt
!cp {best_s} /content/drive/MyDrive/yolov8s_best.pt
